# 01 — Python from zero

**Learner scaffold · RetailFlow dataset 1.0.0**

Course lessons: Day 1, Lesson 2 (60 minutes).

Goal: Calculate S0003 version 1 sales before returns using Python.

Before running: Dataset uploaded; assigned configuration filled in. The calculation steps also run in ordinary Python.

Writes: No files or tables.

Follow the playlist module order. Build Buddy will help you create the missing numbered exercise cells. The instructor has a separate completed reference. Step IDs are stable anchors for that authoring work. Saved outputs are blank unless explicitly verified locally; a passed local check does not imply Databricks execution.


## Your notebook grows through Build Buddy

Configuration and instructor utility cells are supplied so you can focus on the learning task. The numbered exercise cells have no solutions. Read the playlist instructions, attempt each cell, run it, explain its output, then ask for a hint if needed. A blank exercise cell is incomplete work. Do not use Run All expecting this scaffold to be a finished pipeline. The course revisits notebooks: Lesson 5 runs 02-02, 02-03 and 02-05; Lesson 6 then runs 02-04.


## Configuration

Use the same assigned names in every notebook. Each notebook defines its own variables and helpers; it does not depend on another notebook’s Python memory. Persistent tables are the handoff between notebooks.


In [0]:
# Edit these four values in EVERY notebook before running it.
CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "retailflow_files"
TABLE_PREFIX = "rf_retailflow"

import re
for name in [CATALOG, SCHEMA, VOLUME, TABLE_PREFIX]:
    assert re.fullmatch(r"[a-z][a-z0-9_]*", name), "Use lowercase letters, digits, and underscores; start with a letter."
assert not any(name.startswith("your_") for name in [CATALOG, SCHEMA, VOLUME]), "Replace the example names with your assigned names."
assert TABLE_PREFIX != "rf_yourname", "Choose the table prefix assigned to you."
DATASET_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/retailflow-v1.0.0"
print("Dataset:", DATASET_ROOT)
print("Table prefix:", f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_")


Dataset: /Volumes/workspace/default/retailflow_files/retailflow-v1.0.0
Table prefix: workspace.default.rf_retailflow_


## 01-01: Give a value a name

**Core concept Vibe:** A variable holds a value we can reuse.

**Build Buddy action:** Create variables for the sale ID and its quantity.

**Expected checkpoint:** S0003 and quantity 3.


In [0]:
# Build Buddy step 01-01
# A variable holds a value we can reuse.
sale_id = "S0003"
quantity = 3
print("Sale ID:", sale_id)
print("Quantity:", quantity)


Sale ID: S0003
Quantity: 3


## 01-02: Represent money and a percentage

**Core concept Vibe:** Text, whole numbers, and decimal numbers have different uses.

**Build Buddy action:** Represent the unit price and discount. Decimal preserves exact money values.

**Expected checkpoint:** INR 150.00 per Tea 250 g pack; 10 means 10 percent.


In [0]:
# Build Buddy step 01-02
# Decimal preserves exact money values; the discount stays a whole-number percentage.
from decimal import Decimal

product_name = "Tea 250 g pack"
unit_price = Decimal("150.00")  # INR per pack
discount_pct = 10  # 10 percent
print(product_name, "unit price: INR", unit_price)
print("Discount:", discount_pct, "percent")


Tea 250 g pack unit price: INR 150.00
Discount: 10 percent


## 01-03: Multiply quantity by price

**Core concept Vibe:** Multiplication calculates the amount before discounts.

**Build Buddy action:** Calculate gross sales.

**Expected checkpoint:** Gross sales = INR 450.00.


In [0]:
# Build Buddy step 01-03
# Gross sales is quantity multiplied by unit price, before any discount.
gross_sales = quantity * unit_price
print("Gross sales: INR", gross_sales)


Gross sales: INR 450.00


## 01-04: Calculate the discount

**Core concept Vibe:** A percentage means a number out of 100.

**Build Buddy action:** Convert 10 percentage points to a fraction and calculate the discount amount.

**Expected checkpoint:** Discount fraction = 0.1; discount amount = INR 45.00.


In [0]:
# Build Buddy step 01-04
# 10 percent means 10 out of 100, so divide by 100 to get the fraction.
discount_fraction = Decimal(discount_pct) / 100
discount_amount = (gross_sales * discount_fraction).quantize(Decimal("0.01"))
print("Discount fraction:", discount_fraction)
print("Discount amount: INR", discount_amount)


Discount fraction: 0.1
Discount amount: INR 45.00


## 01-05: Subtract the discount

**Core concept Vibe:** Net sales is sales after discounts and before returns.

**Build Buddy action:** Calculate and display net sales.

**Expected checkpoint:** S0003 version 1 net sales = INR 405.00.


In [0]:
# Build Buddy step 01-05
# Net sales: after the discount, before returns.
net_sales = gross_sales - discount_amount
print(f"{sale_id} version 1 net sales: INR {net_sales}")


S0003 version 1 net sales: INR 405.00


## 01-06: Group related values

**Core concept Vibe:** A dictionary gives names to several values in one record.

**Build Buddy action:** Create a dictionary describing this sale calculation.

**Expected checkpoint:** The dictionary holds the same values; no new sale is invented.


In [0]:
# Build Buddy step 01-06
# One dictionary names every value in this sale calculation.
sale_calculation = {
    "sale_id": sale_id,
    "quantity": quantity,
    "unit_price": unit_price,
    "discount_pct": discount_pct,
    "gross_sales": gross_sales,
    "discount_amount": discount_amount,
    "net_sales": net_sales,
}
print(sale_calculation)


{'sale_id': 'S0003', 'quantity': 3, 'unit_price': Decimal('150.00'), 'discount_pct': 10, 'gross_sales': Decimal('450.00'), 'discount_amount': Decimal('45.00'), 'net_sales': Decimal('405.00')}


## 01-07: Verify against the real file

**Core concept Vibe:** An answer should be traceable to its source record.

**Build Buddy action:** Read S0003 from the initial CSV and check the values used above.

**Expected checkpoint:** The manually entered values exactly match the released CSV.


In [0]:
# Build Buddy step 01-07
# Trace the answer back to the released CSV record.
import csv
from pathlib import Path

with (Path(DATASET_ROOT) / "data/sales_batch_01.csv").open(encoding="utf-8", newline="") as stream:
    source_row = next(row for row in csv.DictReader(stream) if row["sale_id"] == sale_id)

print(source_row)
assert int(source_row["quantity"]) == quantity, source_row
assert Decimal(source_row["unit_price"]) == unit_price, source_row
assert Decimal(source_row["discount_pct"]) == Decimal(discount_pct), source_row
assert int(source_row["record_version"]) == 1, source_row
print("PASS: values used above exactly match the released CSV for", sale_id)


{'source_row_id': 'B01-003', 'sale_id': 'S0003', 'sale_date': '2026-01-01', 'store_id': 'ST03', 'product_id': 'P003', 'customer_id': 'C003', 'quantity': '3', 'unit_price': '150.00', 'discount_pct': '10', 'payment_method': 'CASH', 'record_version': '1'}
PASS: values used above exactly match the released CSV for S0003


## Explain it before moving on

What is a variable? Why divide discount_pct by 100? Why are net sales different from profit? Explain why this notebook reports version 1 and does not yet apply the Day 2 correction.

Suggested pacing: 10 minutes for variables, 10 for data types and money, 20 for arithmetic, 10 for the dictionary and source lookup, 10 for practice and explanation. Keep the original cells intact for exact checks; use a scratch cell for other numbers.
